In [0]:
SHOW TABLES IN workspace.default;
DESCRIBE workspace.default.sizwe_sales_lines;

-- TASK 1
SELECT sales_orders.channel,
SUM(CASE
    WHEN sales_orders.order_date <= DATE '2024-06-30'
    THEN sales_lines.line_revenue
    ELSE 0 END) AS revenue_2024,
SUM(CASE
    WHEN sales_orders.order_date > DATE '2024-06-30'
    THEN sales_lines.line_revenue
    ELSE 0 END) AS revenue_2025,
ROUND(
    (SUM(CASE
         WHEN sales_orders.order_date > DATE '2024-06-30'
         THEN sales_lines.line_revenue
         ELSE 0 END)
         -
         SUM(CASE
             WHEN sales_orders.order_date <= DATE'2024-06-30'
             THEN sales_lines.line_revenue
             ELSE 0 END)
        )
        / NULLIF(
            SUM(CASE
                WHEN sales_orders.order_date <= DATE'2024-06-30'
                THEN sales_lines.line_revenue
                ELSE 0 END),
            0

        )
        * 100,

    2
) AS growth_rate
FROM workspace.default.sizwe_sales_orders AS sales_orders
JOIN workspace.default.sizwe_sales_lines AS sales_lines ON 
sales_orders.order_id = sales_lines.order_id
GROUP BY sales_orders.channel
ORDER BY sales_orders.channel;

--TASK 2
SELECT products.product_name, SUM(sales_lines.line_revenue) AS total_revenue,SUM(sales_lines.cogs) AS total_costs,
SUM(sales_lines.line_revenue) - SUM(sales_lines.cogs) AS Gross_Profit,
ROUND((
    SUM(sales_lines.line_revenue) - SUM(sales_lines.cogs)
) / NULLIF (SUM(sales_lines.line_revenue), 0) * 100,
2
) AS Gross_Margin
FROM workspace.default.sizwe_products AS products
JOIN workspace.default.sizwe_sales_lines AS sales_lines
ON sales_lines.sku = products.sku
GROUP BY products.product_name
ORDER BY Gross_Profit DESC;
--House Blend generated the highest gross profit, at approximately R107,148,746.60. Single Origin had the highest gross margin at 44.27%, so the product line with the highest gross profit was not the same as the product line with the highest gross margin.

--TASK 3
SELECT sales_orders.channel,
COUNT(DISTINCT sales_orders.order_id) AS number_of_invoices,
COUNT(CASE WHEN ar_ledger.settlement_date IS NULL THEN 1 
END) AS STILL_OPEN,
COUNT(CASE
      WHEN ar_ledger.settlement_date IS NOT NULL THEN 1
      END) AS SETTLED
FROM workspace.default.sizwe_sales_orders AS sales_orders
JOIN workspace.default.sizwe_ar_ledger AS ar_ledger
ON sales_orders.order_id = ar_ledger.order_id
GROUP BY sales_orders.channel;
--Subscription had 44 open invoices and an outstanding balance of R751,178.89, with an average settlement time of 3.85 days. Wholesale had 3,454 open invoices and R87,848,227.20 outstanding, with an average settlement time of 68.68 days. Wholesale should be investigated first because it has a much higher outstanding balance, more open invoices and a longer average settlement time. Sizwe could review overdue accounts and follow up with customers to improve collections.

--TASK 4
SELECT customers.trading_name, customers.segment, customers.province, reps.rep_name, SUM(sales_lines.line_revenue) AS FY2025_revenue
FROM workspace.default.sizwe_customers AS customers 
JOIN workspace.default.sizwe_sales_orders AS sales_orders
ON customers.customer_id = sales_orders.customer_id
JOIN workspace.default.sizwe_sales_lines AS sales_lines
ON sales_orders.order_id = sales_lines.order_id
JOIN workspace.default.sizwe_reps AS reps
ON sales_orders.rep_id = reps.rep_id
WHERE sales_orders.order_date > DATE '2024-06-30'
GROUP BY customers.trading_name,customers.segment,customers.province, reps.rep_name
ORDER BY FY2025_revenue DESC
LIMIT 10;

--TASK 5
SELECT customers.trading_name, customers.segment, customers.province, SUM(sales_lines.line_revenue) AS FY2025_revenue
FROM workspace.default.sizwe_customers AS customers 
JOIN workspace.default.sizwe_sales_orders AS sales_orders
ON customers.customer_id = sales_orders.customer_id
JOIN workspace.default.sizwe_sales_lines AS sales_lines
ON sales_orders.order_id = sales_lines.order_id
WHERE sales_orders.channel = 'Wholesale'
AND NOT EXISTS (
    SELECT 1
    FROM workspace.default.sizwe_trade_promotions AS promotions
    WHERE promotions.customer_id = customers.customer_id
)
GROUP BY customers.trading_name,customers.segment,customers.province
ORDER BY FY2025_revenue DESC
LIMIT 15;
--I would review Umhlanga Fresh 108, Ridgeway Foods 308 and Kalahari Kitchen 42 for suitable promotional support. They are the three highest-revenue Wholesale customers in the results that have no recorded trade promotions, with revenues of approximately R6.22 million, R4.45 million and R4.45 million respectively. Before offering a promotion, Sizwe should consider each customer's buying pattern, profitability and potential for increased sales or stronger retention.

--TASK 6
SELECT reps.rep_name, reps.territory, reps.target_monthly, AVG(sales_lines.line_revenue) AS FY2025_Monthly_target, AVG(sales_lines.line_revenue) / reps.target_monthly * 100 AS percentage_of_target
FROM workspace.default.sizwe_reps AS reps
JOIN workspace.default.sizwe_sales_orders AS sales_orders
ON reps.rep_id = sales_orders.rep_id
JOIN workspace.default.sizwe_sales_lines AS sales_lines
ON sales_orders.order_id = sales_lines.order_id
WHERE sales_orders.order_date > DATE '2024-06-30'
GROUP BY reps.rep_name, reps.territory, reps.target_monthly
ORDER BY percentage_of_target DESC;

--TASK 7
WITH monthly_total AS (
    SELECT DATE_TRUNC('MONTH', sales_orders.order_date) AS sales_month, SUM(sales_lines.line_revenue) AS Monthly_revenue FROM workspace.default.sizwe_sales_orders AS sales_orders
    JOIN workspace.default.sizwe_sales_lines AS sales_lines
    ON sales_orders.order_id = sales_lines.order_id
    GROUP BY DATE_TRUNC('MONTH', sales_orders.order_date)
),
monthly_comparison AS (
    SELECT sales_month, Monthly_revenue, LAG(Monthly_revenue, 1) OVER (ORDER BY sales_month) AS Previous_month_revenue, AVG(monthly_revenue) OVER (ORDER BY sales_month ROWS
    BETWEEN 2 PRECEDING AND CURRENT ROW ) AS 3_MONTH_AVG
    FROM monthly_totals)
SELECT sales_month, Monthly_revenue, Previous_month_revenue,
ROUND((Monthly_revenue - Previous_month_revenue) / 
NULLIF(Previous_month_revenue, 0) * 100, 2) AS Percentage_change, ROUND(3_MONTH_AVG, 2) AS 3_MONTH_AVG
FROM monthly_comparison
ORDER BY sales_month;
--The monthly results suggest a recurring year-end slowdown. December 2023 was the quietest month in the 24-month results, with revenue of approximately R29.87 million, a month-on-month decrease of 13.09%. Revenue also fell by 12.89% in December 2024, before recovering in January and February 2025. Sizwe could consider whether seasonal customer demand or ordering patterns explain these declines and plan sales activity accordingly.

--TASK 8
WITH product_year_revenue AS (
    SELECT
        p.product_line,
        CASE
            WHEN so.order_date <= DATE '2024-06-30'
                THEN 'FY2024'
            ELSE 'FY2025'
        END AS financial_year,
        SUM(sl.line_revenue) AS revenue,
        SUM(sl.cogs) AS total_cogs
    FROM workspace.default.sizwe_sales_orders AS so
    JOIN workspace.default.sizwe_sales_lines AS sl
        ON so.order_id = sl.order_id
    JOIN workspace.default.sizwe_products AS p
        ON sl.sku = p.sku
    GROUP BY
        p.product_line,
        CASE
            WHEN so.order_date <= DATE '2024-06-30'
                THEN 'FY2024'
            ELSE 'FY2025'
        END
)
SELECT
    product_line,
    financial_year,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        (revenue - total_cogs) / NULLIF(revenue, 0) * 100,
        2
    ) AS gross_margin_pct,
    ROUND(
        revenue / NULLIF(
            SUM(revenue) OVER (PARTITION BY financial_year),
            0
        ) * 100,
        2
    ) AS share_of_total_revenue_pct
FROM product_year_revenue
ORDER BY
    product_line,
    financial_year;
--Overall gross margin fell from 34.7% in FY2024 to 33.0% in FY2025, a decrease of 1.7 percentage points, even though revenue grew by nearly 9%. Bulk Commercial's share of total revenue increased from 25.02% to 33.65%, while its gross margin remained around 19.9% in both years. Because Bulk Commercial has a much lower margin than the other product lines, its increased share of revenue is a plausible contributor to the decline in overall gross margin. This does not establish that the change in product mix was the only cause.


